# Drift Detection with MLflow

Compare a **reference** dataset (the data the model was trained on) with a **new** dataset
and check for data drift:

- **Numeric features** -> Kolmogorov-Smirnov (KS) test
- **Categorical features** -> Chi-Square test

If drift is detected, we retrain the model and save it as `ddmmyy_xgb_car_price_model.pkl`,
logging everything to MLflow.


In [1]:
import datetime
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
import mlflow
import mlflow.xgboost


In [2]:
# `mlflow ui` (no arguments) defaults to sqlite:///mlflow.db in the current directory,
# so logging here means the runs show up with nothing more than:
#
#     cd model_tracking
#     mlflow ui
#
# then open http://127.0.0.1:5000 and pick the "xgb_car_price_drift" experiment.
mlflow.set_tracking_uri("sqlite:///mlflow38.db")
mlflow.set_experiment("xgb_car_price_drift")


2026/09/02 23:20:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/02 23:20:22 INFO mlflow.store.db.utils: Updating database tables
2026/09/02 23:20:22 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/09/02 23:20:22 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/09/02 23:20:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/09/02 23:20:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='file:c:/Users/swath/mlops_scaler/model_tracking/mlruns/2', creation_time=1788370714744, experiment_id='2', last_update_time=1788370714744, lifecycle_stage='active', name='xgb_car_price_drift', tags={}>

## 1. Load the reference and new datasets

The reference CSV lives in the repo's `data/` folder, one level up from this notebook.
Today's "incoming" batch is simulated from it (see below) so the notebook is runnable
out of the box.


### Simulate today's incoming batch

`{TODAY}_cars_24_price.csv` is a copy of the reference data with a small amount of
noise added on purpose: km_driven and age drift up a little, mileage and selling_price
drift down a little, and ~3% of the petrol cars are relabelled as diesel. That is
enough for the KS / Chi-Square tests below to actually fire.


In [6]:
reference_df = pd.read_csv("../data/cars24-car-price-cleaned-new.csv")
new_df = pd.read_csv("020926_cars_24_price.csv")

print(f"reference rows: {len(reference_df)},  new rows: {len(new_df)}")


print(f"reference rows: {len(reference_df)},  new rows: {len(new_df)}")


reference rows: 19820,  new rows: 19820
reference rows: 19820,  new rows: 19820


## 2. Define which features are numeric and which are categorical

We use the model's input features plus the target (`selling_price`).

In [18]:
NUMERIC_FEATURES = ["km_driven", "mileage", "age", "selling_price"]
CATEGORICAL_FEATURES = ["Petrol", "Diesel", "Electric"]

ALPHA = 0.05  # significance level: p-value below this means the distributions differ (drift)


## 3. KS test for numeric features

In [19]:
def ks_drift(reference, new, features, alpha=ALPHA):
    rows = []
    for col in features:
        stat, p_value = ks_2samp(reference[col], new[col])
        rows.append({"feature": col, "test": "KS", "p_value": round(p_value, 4),
                     "drift": p_value < alpha})
    return pd.DataFrame(rows)


numeric_results = ks_drift(reference_df, new_df, NUMERIC_FEATURES)
numeric_results


,feature,test,p_value,drift
0,km_driven,KS,0.0,True
1,mileage,KS,0.0,True
2,age,KS,0.0,True
3,selling_price,KS,0.0,True


## 4. Chi-Square test for categorical features

In [20]:
def chi_square_drift(reference, new, features, alpha=ALPHA):
    rows = []
    for col in features:
        # build a contingency table of category counts in each dataset
        ref_counts = reference[col].value_counts()
        new_counts = new[col].value_counts()
        table = pd.concat([ref_counts, new_counts], axis=1).fillna(0)
        stat, p_value, _, _ = chi2_contingency(table)
        rows.append({"feature": col, "test": "Chi2", "p_value": round(p_value, 4),
                     "drift": p_value < alpha})
    return pd.DataFrame(rows)


categorical_results = chi_square_drift(reference_df, new_df, CATEGORICAL_FEATURES)
categorical_results


,feature,test,p_value,drift
0,Petrol,Chi2,0.0037,True
1,Diesel,Chi2,0.0037,True
2,Electric,Chi2,1.0000,False


## 5. Combine results and decide if drift occurred

In [21]:
drift_report = pd.concat([numeric_results, categorical_results], ignore_index=True)
drift_detected = bool(drift_report["drift"].any())

print(drift_report)
print("Drift detected:", drift_detected)

         feature  test  p_value  drift
0      km_driven    KS   0.0000   True
1        mileage    KS   0.0000   True
2            age    KS   0.0000   True
3  selling_price    KS   0.0000   True
4         Petrol  Chi2   0.0037   True
5         Diesel  Chi2   0.0037   True
6       Electric  Chi2   1.0000  False
Drift detected: True


## 6. If drift is detected, retrain and save a new model

The retrained model is saved as `ddmmyy_xgb_car_price_model.pkl` so that downstream
checks (and our test suite) can detect that a fresh model was produced today.
Everything is logged to MLflow.

In [22]:
MODELS_DIR = Path("..") / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ["km_driven", "mileage", "age", "Petrol", "Diesel", "Electric"]
TARGET = "selling_price"

with mlflow.start_run(run_name=f"drift_check_{TODAY}"):
    # log the drift report
    mlflow.log_param("drift_detected", drift_detected)
    for _, row in drift_report.iterrows():
        mlflow.log_metric(f"pvalue_{row['feature']}", row["p_value"])

    if drift_detected:
        print("Drift detected -> retraining model on the new data...")

        # retrain on reference + new data combined
        combined = pd.concat([reference_df, new_df], ignore_index=True)
        X = combined[FEATURES]
        y = combined[TARGET]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = XGBRegressor(n_estimators=250, learning_rate=0.03, max_depth=20)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        mlflow.log_metric("mape", mape)
        mlflow.log_metric("rmse", rmse)

        model_name = f"{TODAY}_xgb_car_price_model"
        model_path = MODELS_DIR / f"{model_name}.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        mlflow.xgboost.log_model(model, name=model_name)
        print(f"Saved retrained model to {model_path}")
    else:
        print("No drift detected -> keeping the existing model.")


Drift detected -> retraining model on the new data...
Saved retrained model to ..\models\020926_xgb_car_price_model.pkl
